# Lab 1: Build the HR policy index

## Business problem

Employees need answers from several approved HR policies. Before an LLM can answer, the documents must become searchable records that still identify their source and policy section.

## Mission

Turn the HR policy folder into a searchable index and verify that the correct evidence is retrieved for an employee question.

## Exercise 1: Inspect what entered the pipeline

**Mission:** Load every approved policy file and confirm that its text is readable.

**Why it matters:** Missing or damaged text cannot be repaired by embeddings or an LLM. Real PDF and DOCX files require a parser before this point.

In [ ]:
from pathlib import Path

policy_files = sorted(Path("policies").glob("*.md"))

print("Policy files:", len(policy_files))
for policy_file in policy_files:
    text = policy_file.read_text(encoding="utf-8")
    print(policy_file.name, "|", len(text), "characters")

### Inspection checkpoint

Five files should appear. Open one file and confirm that its title, headings, and paragraphs were preserved.

## Exercise 2: Preserve complete policy sections

**Mission:** Split each Markdown document at its `##` section headings.

**Why it matters:** A policy heading gives meaning to the text below it and becomes useful citation metadata.

In [ ]:
sections = []

for policy_file in policy_files:
    policy_text = policy_file.read_text(encoding="utf-8")
    parts = policy_text.split("\n## ")

    for part in parts[1:]:
        heading, text = part.split("\n", 1)
        sections.append({"source": policy_file.name, "section": heading, "text": text.strip()})

for section in sections:
    print(section["source"], "|", section["section"])

### What the code did

`split` created a list wherever a second-level Markdown heading appeared. The first line of each part became its heading. The remaining text became its content.

## Exercise 3: Split only sections that exceed the limit

**Mission:** Apply the selected structure-aware recursive chunking configuration.

The lecture discusses token limits. This beginner lab uses characters so the behavior is easy to inspect. These are different measurements. A production configuration must record which measurement its splitter uses.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

In [ ]:
chunks = []

for section in sections:
    pieces = splitter.split_text(section["text"])

    for position, piece in enumerate(pieces):
        chunks.append({**section, "text": piece, "position": position})

for chunk in chunks:
    print(chunk["section"], "| position", chunk["position"], "|", len(chunk["text"]), "characters")

### Chunking checkpoint

Short sections should remain intact. If a section is split, every resulting chunk should retain the same source and section. Overlap is used only when a split occurs.

## Exercise 4: Inspect one embedding

**Mission:** Convert one chunk into the numeric representation used for similarity search.

**Why it matters:** The index and employee questions must use the same embedding model and compatible dimensions.

**Industry choices:** This lab uses OpenAI `text-embedding-3-small` because it is recognizable in job descriptions and simple to call. Other commonly encountered choices include Cohere Embed, Voyage AI embeddings, Google Gemini embeddings, and open-weight models such as BGE or E5. Choose one provider for an index and do not mix vector dimensions.

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
EMBEDDING_MODEL = "text-embedding-3-small"

response = client.embeddings.create(model=EMBEDDING_MODEL, input=chunks[0]["text"])
vector = response.data[0].embedding

print("Vector dimensions:", len(vector))
print("First 10 values:", vector[:10])

## Exercise 5: Build the vector index

**Mission:** Store every chunk together with the metadata needed for filtering and citations.

**Industry choices:** This lab uses LangChain's `InMemoryVectorStore` so the retrieval steps remain visible. Production teams commonly use PostgreSQL with pgvector, Pinecone, Qdrant, Weaviate, Milvus, or a cloud search service such as Azure AI Search. The choice depends on persistence, scale, hybrid search, access control, and operating model.

In [ ]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

documents = []
for chunk in chunks:
    metadata = {
        "source": chunk["source"],
        "section": chunk["section"],
        "position": chunk["position"],
        "version": "2026.1",
        "status": "current",
    }
    documents.append(Document(page_content=chunk["text"], metadata=metadata))

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vector_store = InMemoryVectorStore.from_documents(documents, embedding=embeddings)
print("Indexed chunks:", len(documents))

## Exercise 6: Retrieve and inspect evidence

**Mission:** Verify which policy evidence would be sent to an LLM.

Change `TOP_K` and inspect the effect. More retrieved chunks can add useful evidence or unrelated material.

In [ ]:
QUESTION = "Can I expense a $300 train ticket without approval?"
TOP_K = 3

results = vector_store.similarity_search_with_score(QUESTION, k=TOP_K)

for document, score in results:
    print("Similarity:", round(score, 3))
    print("Source:", document.metadata["source"])
    print("Section:", document.metadata["section"])
    print(document.page_content)
    print()

## Exercise 7: Record the index configuration

**Mission:** Record enough configuration to rebuild or troubleshoot this index later.

An index depends on the source files, parser, chunk settings, embedding model, and index version.

In [ ]:
INDEX_CONFIG = {
    "index_version": "2026.1",
    "parser": "Markdown headings and paragraphs",
    "chunk_size_characters": CHUNK_SIZE,
    "chunk_overlap_characters": CHUNK_OVERLAP,
    "embedding_model": EMBEDDING_MODEL,
}

for name, value in INDEX_CONFIG.items():
    print(name, "=", value)

### Operations checkpoint

If any of these settings change, rebuild the index and run the retrieval tests again. This is the same idea as recording the software version and configuration before a network change.

## Lab 1 checkpoint

`reimbursements.md` and `Travel approval` should rank first. You can now trace a retrieval result through its embedding configuration, chunk, section, and source.